In [31]:
import pandas as pd
import numpy as np
import json

movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

print(movies.shape)   # expect (4803, 20)
print(credits.shape)  # expect (4803, 4)
print(movies.columns.tolist())  


(4803, 20)
(4803, 4)
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [32]:
# credits has 'movie_id', movies has 'id' — align them
credits = credits.rename(columns={'movie_id': 'id', 'title': 'credit_title'})
df = movies.merge(credits, on='id')

print(df.shape)  # expect (4803, 23)

(4803, 23)


In [33]:
# Films with zero revenue or zero budget are either
df = df[df['revenue'] > 0]
df = df[df['budget'] > 10_000]  

print(f"Remaining rows: {len(df)}")  # expect ~2,400

Remaining rows: 3213


In [34]:
def parse_json_col(val):
    try:
        return json.loads(val.replace("'", '"'))
    except:
        return []
    
def is_franchise_from_keywords(keyword_str):
    keywords = parse_json_col(keyword_str)
    names = [k['name'].lower() for k in keywords]
    franchise_words = ['sequel', 'based on novel', 'spin off', 
                       'prequel', 'part of series', 'franchise']
    return 1 if any(w in names for w in franchise_words) else 0

df['is_franchise'] = df['keywords'].apply(is_franchise_from_keywords)

# Quick check — should flag films like Harry Potter, Fast & Furious etc.
print(df[df['is_franchise']==1][['title']].head(10))
print(df['is_franchise'].value_counts())
print(f"Franchise rate: {df['is_franchise'].mean():.1%}")

                                       title
2                                    Spectre
4                                John Carter
7                    Avengers: Age of Ultron
10                          Superman Returns
15  The Chronicles of Narnia: Prince Caspian
26                Captain America: Civil War
28                            Jurassic World
32                       Alice in Wonderland
36           Transformers: Age of Extinction
38                  The Amazing Spider-Man 2
is_franchise
0    2962
1     251
Name: count, dtype: int64
Franchise rate: 7.8%


In [35]:
# Genres → list of genre names
df['genres_parsed'] = df['genres'].apply(
    lambda x: [g['name'] for g in parse_json_col(x)]
)

# Production companies → list of company names
df['companies_parsed'] = df['production_companies'].apply(
    lambda x: [c['name'] for c in parse_json_col(x)]
)

# Keywords
df['keywords_parsed'] = df['keywords'].apply(
    lambda x: [k['name'] for k in parse_json_col(x)]
)

print(df[['genres_parsed', 'is_franchise']].head(5))

                                   genres_parsed  is_franchise
0  [Action, Adventure, Fantasy, Science Fiction]             0
1                   [Adventure, Fantasy, Action]             0
2                     [Action, Adventure, Crime]             1
3               [Action, Crime, Drama, Thriller]             0
4           [Action, Adventure, Science Fiction]             1


In [37]:

def get_cast_popularity(cast_str, top_n=3):
    cast = parse_json_col(cast_str)
    return sum(c.get('popularity', 0) for c in cast[:top_n])

def get_director(crew_str):
    crew = parse_json_col(crew_str)
    for member in crew:
        if member.get('job') == 'Director':
            return member.get('name', 'Unknown')
    return 'Unknown'

df['director'] = df['crew'].apply(get_director)


In [38]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date'])

df['release_year']  = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
df['release_dow']   = df['release_date'].dt.dayofweek  # 0=Mon, 4=Fri

# Summer (Jun–Aug) and holiday (Nov–Dec) are high-revenue windows
df['is_summer']  = df['release_month'].isin([6, 7, 8]).astype(int)
df['is_holiday'] = df['release_month'].isin([11, 12]).astype(int)

In [39]:
TOP_GENRES = ['Action', 'Comedy', 'Drama', 'Thriller',
              'Animation', 'Horror', 'Romance', 'Adventure']

for genre in TOP_GENRES:
    df[f'genre_{genre}'] = df['genres_parsed'].apply(
        lambda g: 1 if genre in g else 0
    )

In [40]:
KEEP_COLS = [
    'id', 'title',
    'budget', 'revenue', 'runtime', 'popularity',
    'original_language',
    'director', 'is_franchise',
    'release_year', 'release_month', 'release_dow',
    'is_summer', 'is_holiday',
] + [f'genre_{g}' for g in TOP_GENRES]

df_clean = df[KEEP_COLS].copy()

# Fill any remaining nulls
df_clean['runtime'] = df_clean['runtime'].fillna(df_clean['runtime'].median())

# Encode language: English=1, other=0
df_clean['is_english'] = (df_clean['original_language'] == 'en').astype(int)
df_clean = df_clean.drop(columns=['original_language'])

print(df_clean.shape)
print(df_clean.isnull().sum())  # should be all zeros

(3213, 22)
id                 0
title              0
budget             0
revenue            0
runtime            0
popularity         0
director           0
is_franchise       0
release_year       0
release_month      0
release_dow        0
is_summer          0
is_holiday         0
genre_Action       0
genre_Comedy       0
genre_Drama        0
genre_Thriller     0
genre_Animation    0
genre_Horror       0
genre_Romance      0
genre_Adventure    0
is_english         0
dtype: int64


In [42]:
print(df_clean.describe()[['budget', 'revenue', 'runtime']])

# Check a famous film looks right
print(df_clean[df_clean['title'] == 'Avatar'][['budget','revenue','is_franchise']])

             budget       revenue      runtime
count  3.213000e+03  3.213000e+03  3213.000000
mean   4.085689e+07  1.218381e+08   110.803922
std    4.441414e+07  1.865746e+08    20.938299
min    1.200000e+04  7.000000e+00    41.000000
25%    1.100000e+07  1.729238e+07    96.000000
50%    2.500000e+07  5.596900e+07   107.000000
75%    5.500000e+07  1.472988e+08   121.000000
max    3.800000e+08  2.787965e+09   338.000000
      budget     revenue  is_franchise
0  237000000  2787965087             0


In [43]:
df_clean.to_parquet('../data/processed/movies_clean.parquet', index=False)
print("Saved! Shape:", df_clean.shape)  

Saved! Shape: (3213, 22)
